<a href="https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""My lane (Refresh / Content Opportunity Scoring) is best framed as a ranking/scoring task, not plain classification. The end deliverable is a ranked list of pages to review first, ordered by priority — not just a yes/no label. Under the hood this is built on a binary classification model (predicting is_declining_label), but the model's output (a probability) is used as a score to rank pages, then evaluated with ranking-style metrics like Precision@K rather than plain accuracy. This matches how the starter pipeline itself works: a classifier trained on trend_direction, whose probability output becomes the ranking signal for the final refresh queue."""


"My lane (Refresh / Content Opportunity Scoring) is best framed as a ranking/scoring task, not plain classification. The end deliverable is a ranked list of pages to review first, ordered by priority — not just a yes/no label. Under the hood this is built on a binary classification model (predicting is_declining_label), but the model's output (a probability) is used as a score to rank pages, then evaluated with ranking-style metrics like Precision@K rather than plain accuracy. This matches how the starter pipeline itself works: a classifier trained on trend_direction, whose probability output becomes the ranking signal for the final refresh queue."

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Target (current, starter-notebook version): is_declining_label = (trend_direction == "down"), a binary proxy for "this page needs review."

Known weakness: this is a current-window label, not a future outcome — it says a page is declining right now, not that it will decline further. For my capstone I want to move toward a future-window proxy: features from a prior 90-day window predicting decline over the next 30 days, which is a stronger, less circular target. I'm starting with the current-window proxy for this task-framing exercise because it's what the starter data directly supports without extra window-building."""

'Target (current, starter-notebook version): is_declining_label = (trend_direction == "down"), a binary proxy for "this page needs review."\n\nKnown weakness: this is a current-window label, not a future outcome — it says a page is declining right now, not that it will decline further. For my capstone I want to move toward a future-window proxy: features from a prior 90-day window predicting decline over the next 30 days, which is a stronger, less circular target. I\'m starting with the current-window proxy for this task-framing exercise because it\'s what the starter data directly supports without extra window-building.'

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Primary metric: Precision@50 — of the top 50 pages the ranking flags, how many are actually declining? This matches the real use case: a reviewer has limited capacity, so what matters is whether the top of the list is trustworthy, not overall accuracy across all 30,000 pages (most of which nobody will ever look at).

From ML-01, the starter baseline scored Precision@50 = 0.240 and the random forest scored 0.740 — that gap is the entire argument for building a model at all. I'll also track average precision as a secondary metric, since it credits the whole ranking rather than just the top 50, which matters if review capacity changes."""


"Primary metric: Precision@50 — of the top 50 pages the ranking flags, how many are actually declining? This matches the real use case: a reviewer has limited capacity, so what matters is whether the top of the list is trustworthy, not overall accuracy across all 30,000 pages (most of which nobody will ever look at).\n\nFrom ML-01, the starter baseline scored Precision@50 = 0.240 and the random forest scored 0.740 — that gap is the entire argument for building a model at all. I'll also track average precision as a secondary metric, since it credits the whole ranking rather than just the top 50, which matters if review capacity changes."

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/meharalirajar060-codeee/Flyrank_Internship_ML_MAR"
REPO_DIR = "Flyrank_Internship_ML_MAR"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

unit_of_analysis = df[["content_id", "client_id", "impressions_90d", "days_since_last_update",
                         "avg_position", "ctr", "trend_direction"]].head(5)
print(unit_of_analysis)

"""One row = one content page (content_id), scored using its own trailing 90-day window of observed signals — impressions, staleness, position, CTR — plus the client it belongs to (client_id), which matters for validation (client-holdout splits, so a client's pages never appear in both train and test)."""


             content_id          client_id  impressions_90d  \
0  content_304f48230142  client_f369cb89fc             3803   
1  content_a1fb4e703a9e  client_4e07408562            15320   
2  content_9aa793d4d895  client_7f2253d7e2            12581   
3  content_331d6c4de07b  client_19581e27de            11751   
4  content_d99b7a2d90ca  client_3fdba35f04            19140   

   days_since_last_update  avg_position   ctr trend_direction  
0                      20          10.6  0.76            down  
1                      25          20.3  0.05            down  
2                      20          36.5  0.09            down  
3                      22           6.2  0.49          stable  
4                      14          44.0  0.13            down  


"One row = one content page (content_id), scored using its own trailing 90-day window of observed signals — impressions, staleness, position, CTR — plus the client it belongs to (client_id), which matters for validation (client-holdout splits, so a client's pages never appear in both train and test)."

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""I already have direct evidence for this from ML-01 and ML-02. The starter's fixed rule (stale >= 180 days AND impressions_90d >= 500) only ever fires on 0.1% of pages — because staleness alone (180+ days) already only applies to 0.6% of the dataset, since the 75th percentile for days_since_last_update is just 104 days. A single hard threshold like this misses almost the entire declining population (54.2% of all pages).

A learned model doesn't need one hard cutoff — it can weigh staleness, position, impressions, and CTR together in graded combination, which is exactly why the random forest reached Precision@50 = 0.740 versus the rule's 0.240 (a ~3.1x lift). This is a case where the relationships between signals aren't simply additive or threshold-based, which is precisely the kind of problem ML is suited for over a fixed rule."""


"I already have direct evidence for this from ML-01 and ML-02. The starter's fixed rule (stale >= 180 days AND impressions_90d >= 500) only ever fires on 0.1% of pages — because staleness alone (180+ days) already only applies to 0.6% of the dataset, since the 75th percentile for days_since_last_update is just 104 days. A single hard threshold like this misses almost the entire declining population (54.2% of all pages).\n\nA learned model doesn't need one hard cutoff — it can weigh staleness, position, impressions, and CTR together in graded combination, which is exactly why the random forest reached Precision@50 = 0.740 versus the rule's 0.240 (a ~3.1x lift). This is a case where the relationships between signals aren't simply additive or threshold-based, which is precisely the kind of problem ML is suited for over a fixed rule."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.